Install required libraries

In [ ]:
!pip install faiss-cpu sentence-transformers transformers PyPDF2

In [ ]:
import os
from pathlib import Path

HOME = Path.home()

MODELS_DIR = HOME / "ai_models"
UPLOADS_DIR = HOME / "ai_uploads"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(UPLOADS_DIR, exist_ok=True)

## Upload files to colab

In **JupyterLab**, 

1. Select **File Browser** on left.
2. Browse to the **UPLOADS_DIR** directory
3. Upload several PDF files

Suggested files:
 - NIST CSF - https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf
 - NIST 800-53 - https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-53r5.pdf
 - OWASP Top 10 - https://github.com/455a0203-e7d6-47dd-b427-37da96ef0718
 - Cisco IOS XE configuration guides
 - OSPF configuration guides - https://www.cisco.com/c/en/us/td/docs/ios-xml/ios/iproute_ospf/configuration/xe-16/iro-xe-16-book.pdf
 - BGP configuration guides
 - VLAN configuration guides
 - NAT configuration guides
 - Azure Virtual Networks
 - Azure VPN Gateway
 - Azure Load Balancer
 - Azure Availability Zones
 - systemd docs
 - ssh docs - https://docs.ssh.com/manuals/client-user/71/Client-user.pdf
 - firewalld docs
 - SELinux documentation
 - RAG paper - https://arxiv.org/pdf/2312.10997
 - Transformer paper
 - Vector database article - https://arxiv.org/pdf/2310.11703
 - Embeddings article

## Load the Base LLM

In [1]:
# Check for GPU (cuda) or CPU
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3),
            1
        ),
        "GB"
    )

CUDA: True
GPU: Tesla T4
Memory: 14.6 GB


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## Beginner
# Slightly larger, better quality (needs GPU ideally)
model_name = "google/flan-t5-base"

# Even smaller (if students are on CPU only and struggling)
# model_name = "google/flan-t5-small"  # keep this as default

# For extra  experiments, there is also
# model_name = "facebook/bart-base"  # medium size; good if GPU present

## Intermediate
# model_name = "BAAI/bge-base-en-v1.5"
# model_name = "Qwen/Qwen2.5-1.5B-Instruct"

## Advanced
# model_name = "BAAI/bge-large-en-v1.5"
# model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir = MODELS_DIR
)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    cache_dir = MODELS_DIR
).to(DEVICE)

In [ ]:
# Confirm device running model - GPU/CUDA or CPU
print(f"Model running on : {next(model.parameters()).device}")

Prepare questions and ask - Before RAG

In [ ]:
questions = [
    "What are the main components of the Solar System?",
    "How does the Internet work at a high level?",
    "What are some applications of artificial intelligence?",
    "What are the 2027 FSA annual maximum?"
]

In [ ]:
questions = [
    "What is a planet?",
    "What is a natural satellite?",
    "Which planets have ring systems?",
    "What objects are included in the Solar System?",
    "How are planets different from small Solar System bodies?",
    "Why are natural satellites important within the Solar System?",
    "What types of objects orbit the Sun?",
    "What are the major categories of objects found in the Solar System?",
    "Besides planets, what other celestial bodies exist in the Solar System?",
    "Compare planets, moons, and small Solar System bodies.",
    "Describe the structure of the Solar System and the objects within it."
]

In [ ]:
questions = [
    "How do I enable OSPF on an interface?",
    "What is the difference between OSPF area 0 and a stub area?",
    "What Cisco commands verify BGP neighbors?",
    "What is the principle of least privilege?",
    "Which CIS control addresses vulnerability management?",
    "What are the major OWASP web application risks?",
    "What is the difference between a Load Balancer and an Application Gateway?",
    "How does zone redundancy work?",
    "What are the components of a virtual network?",
    "How do I restart a service?",
    "How does SELinux differ from standard Unix permissions?",
    "What is the purpose of systemd?",
    "What are embeddings?",
    "How does retrieval augmentation work?",
    "What is the purpose of chunking?",
    "Why does overlap improve retrieval?"
]

In [ ]:
def ask(model, question):
    inputs = tokenizer(question, return_tensors="pt").to(DEVICE)
    outputs = model.generate(**inputs, max_new_tokens=300, num_beams=4)
    print("Q:", question)
    print("A:", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("-" * 80)

Answer all questions.

Include total runtime (wall time) at end

In [ ]:
%%time

for q in questions:
    ask(model, q)

## Extract text from PDF

In [ ]:
import os
import PyPDF2

documents = []

for filename in os.listdir(UPLOADS_DIR):
    if not filename.lower().endswith(".pdf"):
        continue

    filepath = os.path.join(UPLOADS_DIR, filename)

    with open(filepath, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        text = ""

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        documents.append((filename, text))

print("Documents loaded:")
for filename, _ in documents:
    print(filename)

## Build a Vector Store from uploaded documents

| Chunk Size | Usage |
| -- | -- |
| 100-300 chars | Usually too small |
| 500-1000 chars | Common for simple RAG labs |
| 1000-2000 chars | Very common production setting |
| 2000-4000 chars | Larger context retrieval |
| 4000+ chars | Often becomes inefficient |



In [ ]:
chunks = []

# chunk on newline
# for (filename, text) in documents:
#     for paragraph in text.split("\n"):
#         paragraph = paragraph.strip()
#         if len(paragraph) > 120:  # ignore tiny lines
            # chunks.append({
            #     "source": filename,
            #     "text": paragraph
            # })

# chunk on 500 bytes - tune or or down as needed
chunk_size = 500
overlap = 200
for filename, text in documents:
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]

        if len(chunk.strip()) > 100:
            chunks.append({
                "source": filename,
                "text": chunk
            })

Choose the embedding model (Only choose one)

In [ ]:
from sentence_transformers import SentenceTransformer

## Beginner
# Baseline (what you’re using now)
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    cache_folder = MODELS_DIR,
    device = DEVICE
)

# Alternative 1: slightly better quality, still small
# embedder = SentenceTransformer(
#     "sentence-transformers/all-mpnet-base-v2",
#     cache_folder = MODELS_DIR,
#     device = DEVICE
# ))

# Alternative 2: e5-style embeddings (good for RAG)
# embedder = SentenceTransformer(
#     "intfloat/e5-small-v2",
#     cache_folder = MODELS_DIR,
#     device = DEVICE
# ))

## Intermediate
# Alternative 3: BGE small (popular in RAG examples)
# embedder = SentenceTransformer(
#     "BAAI/bge-small-en-v1.5",
#     cache_folder = MODELS_DIR,
#     device = DEVICE
# ))

# Alternative 4: mpnet (larger and slower)
# embedder = SentenceTransformer(
#     "sentence-transformers/all-mpnet-base-v2",
#     cache_folder = MODELS_DIR,
#     device = DEVICE
# )

chunk_embeddings = embedder.encode(
    [chunk["text"] for chunk in chunks]
)

In [ ]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

Setup Retrieval Function

In [ ]:
# how many chunks to evaluate
# adjust k as needed from 3-10. 5-8 suggested

def retrieve(question, k=5):
    q_embed = embedder.encode([question])
    distances, indices = index.search(
        np.array(q_embed), 
        k
    )
    return [
        (chunks[i], distances[0][pos])
        for pos, i in enumerate(indices[0])
    ]

Combined RAG Answer Generator

In [ ]:
def ask_rag(model, question, k=3, SHOW_RETRIEVED_CHUNKS = False):
    retrieved = retrieve(question, k)
    context = "\n".join(
        chunk["text"]
        for chunk in retrieved
    )

    if SHOW_RETRIEVED_CHUNKS:
        print("\nRetrieved Chunks")
        print("-"*50)
    
        for i, chunk in enumerate(retrieved, 1):
            print(f"[Chunk {i}]")
            print(f"Source: {chunk['source']}")
            print(chunk["text"][:300])
            print()

    prompt = (
        "Use the context below to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outputs = model.generate(**inputs, max_new_tokens=300, num_beams=4)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("Q:", question)
    print("A:", answer)
    
    sources = sorted(
        set(chunk["source"] for chunk in retrieved)
    )
    
    print("\nSources:")
    for source in sources:
        print("-", source)
    print("-" * 80, "\n")

Ask same questions - with RAG

In [ ]:
%%time

for q in questions:
    ask_rag(model, q)

## Ask both models simultaneously

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import time

question_box = widgets.Text(
    placeholder="Ask a question...",
    description="Question:",
    layout=widgets.Layout(width='80%')
)

button = widgets.Button(
    description="Ask"
)

output = widgets.Output()

def submit_question(_):

    question = question_box.value.strip()

    if not question:
        return

    with output:
        output.clear_output()

        print("=" * 70)
        print("QUESTION")
        print("=" * 70)
        print(question)
        print()

        print("=" * 70)
        print("LLM ONLY")
        print("=" * 70)

        start = time.time()
        
        try:
            ask(model, question)
        except Exception as e:
            print("Error:", e)

        llm_time = time.time() - start
        print(f"\nRuntime: {llm_time:.2f} sec")

        print()
        print("=" * 70)
        print("RAG")
        print("=" * 70)

        start = time.time()
        
        try:
            ask_rag(model, question, SHOW_RETRIEVED_CHUNKS=True)
        except Exception as e:
            print("Error:", e)

        # Get timing
        rag_time = time.time() - start
        print(f"\nRuntime: {rag_time:.2f} sec")

        #
        # Comparison
        #
        print()
        print("=" * 70)
        print("COMPARISON")
        print("=" * 70)

        print(f"LLM Only : {llm_time:.2f} sec")
        print(f"RAG      : {rag_time:.2f} sec")
        print(f"Overhead : {rag_time - llm_time:.2f} sec")

button.on_click(submit_question)

display(question_box)
display(button)
display(output)

# Cleanup files when finished

Only clean up temporary files when finished. Models and files will need to be redownloaded if used in the future.

In [ ]:
# Cleanup
# import shutil

# shutil.rmtree(MODELS_DIR)
# shutil.rmtree(UPLOADS_DIR)

# print("Temporary files removed")